In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import requests
import pandas as pd
from datetime import datetime
import time

# definicao do caminho do schema

catalogo = 'medalhao_credit'
bronze_db_name = 'bronze_credit'
silver_db_name = 'silver_credit' 

In [0]:
%sql
USE CATALOG medalhao_credit;
USE SCHEMA silver_credit;

In [0]:
# chamados_hora na camada bronze

df_ft_chamados_hora = spark.table(f'{catalogo}.{bronze_db_name}.chamados_hora')
df_ft_chamados_hora.limit(5).display()

# chamados_hora na camada silver

# mudando as colunas para letras minusculas
df_ft_chamados_hora = df_ft_chamados_hora.select([F.col(c).alias(c.lower()) for c in df_ft_chamados_hora.columns])

# convertendo colunas que começam com 'hora' para timestamp

hora_cols = [c for c in df_ft_chamados_hora.columns if c.startswith('hora')]

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.regexp_replace(F.col(col), r' �s ', ' ')
    )

for col in hora_cols:
    df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
        col, F.to_timestamp(col, 'dd/MM/yyyy HH:mm:ss')
    )

# adicionando colunas [tempo_espera_seg, tempo_atendimento_seg, diff_abertura_ingestao_seg]

df_ft_chamados_hora = df_ft_chamados_hora.withColumn(
    'tempo_espera_seg',
    (F.col('hora_inicio_atendimento').cast('long') - F.col('hora_abertura_chamado').cast('long'))
).withColumn(
    'tempo_atendimento_seg',
    (F.col('hora_finalizacao_atendimento').cast('long') - F.col('hora_inicio_atendimento').cast('long'))
).withColumn(
    'diff_abertura_ingestao_seg',
    (F.col('data_ingestao').cast('long') - F.col('hora_abertura_chamado').cast('long'))
)

# linhas em chamados_hora

print(f'linhas em chamados_hora: {df_ft_chamados_hora.count()}')

# drop valores null

df_ft_chamados_hora_null = df_ft_chamados_hora.filter(
    F.col('hora_abertura_chamado').isNull() |
    F.col('hora_inicio_atendimento').isNull() |
    F.col('hora_finalizacao_atendimento').isNull() |
    F.col('data_ingestao').isNull()
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_null)

print(f'linhas com val nulos: {df_ft_chamados_hora_null.count()}')

# drop linhas onde o tempo e inconsistente

df_ft_chamados_hora_tempo_dif = df_ft_chamados_hora.filter(
    (F.col('tempo_espera_seg') < 0) |
    (F.col('tempo_atendimento_seg') < 0) |
    (F.col('diff_abertura_ingestao_seg') < 0)
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_tempo_dif)

print(f'linhas com tempo inconsistente: {df_ft_chamados_hora_tempo_dif.count()}')

# drop valores irregulares em id_cliente e id_chamado

df_ft_chamados_hora_irreg = df_ft_chamados_hora.filter(
    F.col('id_cliente').isNull() &
    F.col('id_chamado').isNull() &
    ~F.col('id_cliente').rlike(r'^[0-9]+$') &
    ~F.col('id_chamado').rlike(r'^[0-9]+$')
)

df_ft_chamados_hora = df_ft_chamados_hora.subtract(df_ft_chamados_hora_irreg)

print(f'linhas com id_cliente, id_chamado irregulares: {df_ft_chamados_hora_irreg.count()}')

print(f'linhas em chamados_hora apos remocao: {df_ft_chamados_hora.count()}')

# salvando na camada silver
df_ft_chamados_hora.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_hora')

df = spark.table(f'{catalogo}.{silver_db_name}.chamados_hora')
df.limit(5).display()

In [0]:
# leitura das tabelas

nomes_tabelas = [
  "base_atendentes", # id_atendente
  "base_motivos", # nome_motivo
  "canais", # nome_canal
  "chamados", # tabela usada como base para os joins
  "chamados_hora", # id_chamado
  "clientes", # id_cliente
  "custos", # id_chamado
  "pesquisa_satisfacao" # id_chamado
]

df_base_atendentes = spark.table(f"{catalogo}.{silver_db_name}.base_atendentes")
df_base_motivos = spark.table(f"{catalogo}.{silver_db_name}.base_motivos")
df_canais = spark.table(f"{catalogo}.{silver_db_name}.canais")
df_chamados = spark.table(f"{catalogo}.{silver_db_name}.chamados")
df_chamados_hora = spark.table(f"{catalogo}.{silver_db_name}.chamados_hora")
df_clientes = spark.table(f"{catalogo}.{silver_db_name}.clientes")
df_custos = spark.table(f"{catalogo}.{silver_db_name}.custos")
df_pesquisa_satisfacao = spark.table(f"{catalogo}.{silver_db_name}.pesquisa_satisfacao")

In [0]:
df_base_motivos.limit(5).display()

In [0]:
# criacao da tabela chamados_geral

df_chamados_geral = (
    df_chamados
    .join(df_chamados_hora, "id_chamado", "right")
    .join(df_base_atendentes, "id_atendente", "left")
    .join(df_base_motivos, df_chamados["motivo"] == df_base_motivos["nome_motivo"], "left")
    .join(df_canais, df_chamados["canal"] == df_canais["nome_canal"], "left")
    .join(df_clientes, "id_cliente", "left")
    .join(df_custos, "id_chamado", "left")
    .join(df_pesquisa_satisfacao, "id_chamado", "left")
    .select(
        "id_chamado",
        df_chamados["id_cliente"].alias("id_cliente"),
        "motivo",
        "categoria", # tudo nulo
        "categoria_nota",
        "nota_atendimento",
        "criticidade", # tudo nulo
        "canal",
        "status_canal",
        "resolvido",
        df_chamados_hora["hora_abertura_chamado"],
        df_chamados_hora["hora_inicio_atendimento"],
        df_chamados_hora["hora_finalizacao_atendimento"],
        "tempo_espera_segundos",
        "tempo_atendimento_segundos",
        "id_atendente",
        "nome_atendente",
        "nivel_atendimento",
        "valor_custo",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade"
    ).orderBy('id_chamado')
)

display(df_chamados_geral)

In [0]:
df_chamados_geral.write.mode('overwrite').saveAsTable(f'{catalogo}.{silver_db_name}.chamados_geral')